In [0]:
%run ../delta_function

In [0]:
#bibliothèques à importer
import pandas as pd 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, floor
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, coalesce
from pyspark.sql.window import Window
from pyspark.sql.functions import col, expr
from pyspark.sql.types import FloatType
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev' #dev
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'parameters'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
catalog_prd = f"""mal_maite_{current_environment}"""

In [0]:
nogent1_features = spark.table(f"{catalog_prd}.nogent1.features") 
nogent2_features = spark.table(f"{catalog_prd}.nogent2.features") 
rouen1_features = spark.table(f"{catalog_prd}.rouen1.features") 
strasbourg2_features = spark.table(f"{catalog_prd}.strasbourg2.features") 
prouvy1_features = spark.table(f"{catalog_prd}.prouvy1.features") 
polisy1_features = spark.table(f"{catalog_prd}.polisy1.features") 
buzau1_features = spark.table(f"{catalog_prd}.buzau1.features")
bolelemi1_features = spark.table(f"{catalog_prd}.bolelemi1.features")

In [0]:
df_features = nogent1_features.unionByName(nogent2_features).unionByName(rouen1_features).unionByName(strasbourg2_features).unionByName(prouvy1_features).unionByName(polisy1_features).unionByName(buzau1_features).unionByName(bolelemi1_features)

df_features_filtered = df_features.filter(
    (F.col("is_controllable") == True)
    & (F.col("deleted") == False)
)

display(df_features_filtered)

In [0]:
# Colonnes à convertir de decimal -> float
decimal_cols = ["min_value", "max_value", "step"]

# Conversion
df_features_converted = df_features_filtered.select(
    *[
        col(c).cast(FloatType()).alias(c) if c in decimal_cols else col(c)
        for c in df_features_filtered.columns
    ]
)

IMPORT

In [0]:
table_features_min_max = df_features_converted.select(
    "production_line",
    "feature_reference",
    "min_value",
    "max_value",
    "is_controllable",
    "deleted",
    "is_categorical",
    "is_stratification",
    "step",
    "activity"
)

In [0]:
current_process= "fact_features_min_max"

In [0]:
target_fact_features_min_max = current_catalog +"."+current_schema+"."+current_process
print(target_fact_features_min_max)

In [0]:
all_columns =  table_features_min_max.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'production_line'
    ,'feature_reference']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    table_features_min_max, 
    target_fact_features_min_max, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )